# Monitoring and Retraining - Practical Examples
## Learn how to monitor and maintain ML models in production

## Part 1: Model Performance Monitoring

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

print("=== MODEL MONITORING ===")
print()

# Simulate model performance over 90 days
days = pd.date_range(start='2024-05-01', periods=30, freq='D')
accuracy = [0.97, 0.97, 0.96, 0.96, 0.95, 0.95, 0.95, 0.94, 0.94, 0.93,
            0.93, 0.92, 0.92, 0.91, 0.91, 0.90, 0.90, 0.89, 0.88, 0.88,
            0.87, 0.86, 0.85, 0.85, 0.84, 0.83, 0.82, 0.81, 0.80, 0.79]

monitoring_df = pd.DataFrame({
    'Date': days,
    'Accuracy': accuracy,
    'Predictions': np.random.randint(1000, 5000, 30),
    'Errors': np.random.randint(50, 500, 30)
})

monitoring_df['Error_Rate'] = monitoring_df['Errors'] / monitoring_df['Predictions']

print("Daily Model Performance Metrics:")
print("-" * 80)
print(monitoring_df.head(10).to_string(index=False))
print("...")
print(monitoring_df.tail(5).to_string(index=False))

In [ ]:
print("\n=== MONITORING ANALYSIS ===")
print()

# Analyze performance trend
accuracy_change = ((monitoring_df['Accuracy'].iloc[-1] - monitoring_df['Accuracy'].iloc[0]) / 
                   monitoring_df['Accuracy'].iloc[0] * 100)

print(f"Accuracy Change: {accuracy_change:.1f}%")
print(f"Initial Accuracy: {monitoring_df['Accuracy'].iloc[0]:.2%}")
print(f"Current Accuracy: {monitoring_df['Accuracy'].iloc[-1]:.2%}")
print(f"Accuracy Drop: {monitoring_df['Accuracy'].iloc[0] - monitoring_df['Accuracy'].iloc[-1]:.2%}")
print()

# Alert conditions
alert_threshold = 0.90
if monitoring_df['Accuracy'].iloc[-1] < alert_threshold:
    print(f"⚠️ ALERT: Accuracy {monitoring_df['Accuracy'].iloc[-1]:.2%} below threshold {alert_threshold:.2%}")
    print("ACTION: Trigger retraining process")
else:
    print("✓ Model performing normally")

## Part 2: Detecting Data Drift

In [ ]:
print("\n=== DATA DRIFT DETECTION ===")
print()

from scipy import stats

# Historical baseline (training data)
baseline_feature = np.random.normal(loc=100, scale=15, size=1000)

# Current production data (no drift)
no_drift_data = np.random.normal(loc=100.5, scale=15.5, size=500)

# Current production data (with drift)
drift_data = np.random.normal(loc=120, scale=20, size=500)

print("Scenario 1: No Data Drift")
print("-" * 50)
print(f"Baseline mean: {baseline_feature.mean():.1f}")
print(f"Current mean: {no_drift_data.mean():.1f}")
percent_change_1 = abs(no_drift_data.mean() - baseline_feature.mean()) / baseline_feature.mean() * 100
print(f"Change: {percent_change_1:.1f}%")
if percent_change_1 < 5:
    print("✓ No significant drift detected")
else:
    print("⚠ Drift detected")

print()
print("Scenario 2: Data Drift Detected")
print("-" * 50)
print(f"Baseline mean: {baseline_feature.mean():.1f}")
print(f"Current mean: {drift_data.mean():.1f}")
percent_change_2 = abs(drift_data.mean() - baseline_feature.mean()) / baseline_feature.mean() * 100
print(f"Change: {percent_change_2:.1f}%")
if percent_change_2 > 10:
    print("❌ Significant drift detected!")
    print("ACTION: Investigate data changes and retrain model")

## Part 3: Retraining Decision Logic

In [ ]:
print("\n=== RETRAINING DECISION LOGIC ===")
print()

def should_retrain(metrics):
    """
    Determine if model should be retrained
    Returns: (bool, reason)
    """
    
    reasons = []
    
    # Condition 1: Accuracy below threshold
    if metrics['accuracy'] < 0.90:
        reasons.append(f"Accuracy {metrics['accuracy']:.2%} below 90% threshold")
    
    # Condition 2: Error rate too high
    if metrics['error_rate'] > 0.10:
        reasons.append(f"Error rate {metrics['error_rate']:.2%} above 10%")
    
    # Condition 3: Data drift detected
    if metrics['data_drift_detected']:
        reasons.append("Data drift detected")
    
    # Condition 4: Time-based retraining
    if metrics['days_since_retrain'] > 30:
        reasons.append(f"Last retrain {metrics['days_since_retrain']} days ago (30 day limit)")
    
    should_retrain = len(reasons) > 0
    reason_text = " + ".join(reasons) if reasons else "Model performing normally"
    
    return should_retrain, reason_text

# Test different scenarios
scenarios = [
    {
        'name': 'Scenario 1: Normal Performance',
        'metrics': {
            'accuracy': 0.95,
            'error_rate': 0.05,
            'data_drift_detected': False,
            'days_since_retrain': 15
        }
    },
    {
        'name': 'Scenario 2: Accuracy Drops',
        'metrics': {
            'accuracy': 0.88,
            'error_rate': 0.08,
            'data_drift_detected': False,
            'days_since_retrain': 20
        }
    },
    {
        'name': 'Scenario 3: Data Drift',
        'metrics': {
            'accuracy': 0.92,
            'error_rate': 0.06,
            'data_drift_detected': True,
            'days_since_retrain': 10
        }
    },
    {
        'name': 'Scenario 4: Time-Based Retraining',
        'metrics': {
            'accuracy': 0.93,
            'error_rate': 0.05,
            'data_drift_detected': False,
            'days_since_retrain': 35
        }
    },
]

for scenario in scenarios:
    print(scenario['name'])
    print("-" * 60)
    retrain, reason = should_retrain(scenario['metrics'])
    
    if retrain:
        print(f"❌ RETRAIN NEEDED")
        print(f"Reason: {reason}")
    else:
        print(f"✓ NO RETRAIN NEEDED")
        print(f"Status: {reason}")
    print()

## Part 4: Automated Retraining Pipeline

In [ ]:
print("=== AUTOMATED RETRAINING PIPELINE ===")
print()

print("Retraining Workflow:")
print("-" * 60)

steps = [
    ("1. Collect New Data", "Gather last 7 days of data"),
    ("2. Validate Data Quality", "Check for missing values, outliers"),
    ("3. Preprocessing", "Clean and prepare data"),
    ("4. Feature Engineering", "Create features"),
    ("5. Train New Model", "Train with old + new data"),
    ("6. Evaluate Performance", "Calculate metrics on test set"),
    ("7. Compare with Current", "Is new model better?"),
    ("8a. Deploy (if better)", "Backup old, deploy new"),
    ("8b. Keep Current (if worse)", "Continue with current model"),
    ("9. Monitor New Model", "Resume monitoring"),
]

for step, description in steps:
    print(f"{step:25} → {description}")

print()
print("Automation: This entire process can be triggered automatically!")
print("Tools: Apache Airflow, Kubeflow, GitHub Actions")

## Part 5: Monitoring Dashboard

In [ ]:
print("\n=== MONITORING DASHBOARD ===")
print()

print("""
╔════════════════════════════════════════════════════════════════╗
║        ML MODEL MONITORING DASHBOARD                           ║
║        Model: Spam Detector v2.1                               ║
║        Status: HEALTHY ✓                                        ║
╚════════════════════════════════════════════════════════════════╝

REST-OF-DAY METRICS
─────────────────────────────────────────────────────────────────
Accuracy:              96.2%  (Target: > 90%)  ✓
Precision:             95.8%  (Target: > 90%)  ✓
Recall:                95.1%  (Target: > 90%)  ✓
F1-Score:              95.45% (Target: > 90%)  ✓
Error Rate:            3.8%   (Target: < 10%)  ✓

SYSTEM METRICS
─────────────────────────────────────────────────────────────────
Predictions Today:     48,234
Avg Latency:          45ms   (Target: < 100ms)  ✓
Uptime:               99.97% (Target: > 99%)   ✓
Errors:               145    (0.3%)             ✓

DATA QUALITY
─────────────────────────────────────────────────────────────────
Data Drift:           NORMAL  ✓
Missing Values:       0       ✓
Outliers Detected:    2       ⚠
Class Distribution:   Balanced ✓

MODEL INFO
─────────────────────────────────────────────────────────────────
Version:              v2.1
Deployed:             2024-05-15
Last Retrained:       2024-05-10 (5 days ago)
Next Scheduled:       2024-05-25

ALERTS
─────────────────────────────────────────────────────────────────
No active alerts

RECENT ACTIONS
─────────────────────────────────────────────────────────────────
2024-05-18 10:30 - Model performing normally
2024-05-17 15:00 - Minor outliers detected (no action needed)
2024-05-16 09:00 - Routine monitoring check passed
""")

## Summary

1. **Monitoring** - Continuous performance tracking
2. **Drift Detection** - Catch data changes early
3. **Retraining Triggers** - Multiple conditions for retraining
4. **Automation** - Entire process can be automated
5. **Dashboards** - Real-time visibility

Congratulations! You completed the MLOps Learning Module!

## Key Takeaways

✓ MLOps is essential for production ML
✓ Deployment is not the end - monitoring is continuous
✓ Data drift is real and must be detected
✓ Retraining keeps models accurate
✓ Automation prevents manual errors

**Next Steps:**
- Implement these concepts in your projects
- Start with simple monitoring
- Gradually add automation
- Build production-ready ML systems!